# 8×8 LED Matrix CNN (Image → 64‑element board array)

This notebook trains a convolutional neural network (CNN) that takes an **image of an 8×8 LED matrix** and predicts the corresponding **64‑element board array**:

- `0` = LED off  
- `1` = red LED  
- `2` = blue LED  

The labels are loaded from the provided CSV file (one row per image, 64 columns per board). The images are read from a folder, matched in order to the CSV rows, and used to train a CNN that outputs an 8×8×3 tensor (3 classes per LED) which is then interpreted as a 64‑element array.


## 1. Setup

If your images / CSV are stored in Google Drive, first mount Drive and set the paths below. Otherwise, you can upload the files directly to Colab and leave the paths as `/content/...`.

In [ ]:
# OPTIONAL: Mount Google Drive if your data is stored there
USE_DRIVE = False  # change to True if you want to mount Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

import os, glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## 2. Paths and hyperparameters

- `CSV_PATH`: path to the CSV file that contains all 64‑element arrays (one row per image).
- `IMAGE_DIR`: folder containing all LED matrix images.

The code assumes that **sorting the image filenames** gives the same order as the rows in the CSV. If your pairing is different, you can modify how `image_paths` is built (for example using an index column in the CSV).

In [ ]:
# === TODO: Set these to your actual paths ===
CSV_PATH = "/content/boards20.csv"        # e.g. after uploading in Colab
IMAGE_DIR = "/content/led_images"        # folder with your LED matrix images

# Image / training hyperparameters
IMG_SIZE = 128    # size to which images will be resized (IMG_SIZE x IMG_SIZE)
BATCH_SIZE = 32
EPOCHS = 25

# Random seed for reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 3. Load the 64‑element board labels from CSV

The CSV has shape `(N_samples, 64)` where each value is `0`, `1`, or `2`.  
We reshape each 64‑element row into an 8×8 board and keep it as integer labels for the CNN.

In [ ]:
# Load CSV of board arrays
df = pd.read_csv(CSV_PATH)
print("CSV shape:", df.shape)
print("First few rows:")
display(df.head())

# Convert to NumPy array of ints (N, 64)
boards_flat = df.to_numpy(dtype=np.int32)
num_samples, num_cells = boards_flat.shape
assert num_cells == 64, f"Expected 64 columns, got {num_cells}"

# Reshape to (N, 8, 8) for spatial layout
boards_8x8 = boards_flat.reshape(-1, 8, 8)
print("Labels reshaped to:", boards_8x8.shape)

## 4. Load image file paths and match to labels

We gather all image file paths in `IMAGE_DIR`, sort them, and ensure that the number of images matches the number of rows in the CSV.

Supported image extensions: `png`, `jpg`, `jpeg`, `bmp`.

In [ ]:
# Collect all image paths
extensions = ["*.png", "*.jpg", "*.jpeg", "*.bmp"]
image_paths = []
for ext in extensions:
    image_paths.extend(glob.glob(os.path.join(IMAGE_DIR, ext)))

image_paths = sorted(image_paths)
print(f"Found {len(image_paths)} image files.")
if len(image_paths) == 0:
    raise ValueError("No images found. Check IMAGE_DIR and file extensions.")

if len(image_paths) != num_samples:
    print("WARNING: Number of images does not match number of rows in CSV!")
    print("  Images:", len(image_paths))
    print("  CSV rows:", num_samples)
else:
    print("Image count matches CSV rows.")

## 5. Train/validation split

We split the dataset into training and validation sets using a shuffled index split (80% / 20%).

In [ ]:
indices = np.arange(num_samples)
np.random.shuffle(indices)

val_fraction = 0.2
val_size = int(num_samples * val_fraction)

val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_paths = [image_paths[i] for i in train_indices]
val_paths = [image_paths[i] for i in val_indices]
train_labels = boards_8x8[train_indices]
val_labels = boards_8x8[val_indices]

print("Train size:", len(train_paths))
print("Val size:", len(val_paths))

## 6. Build `tf.data` pipelines

We define a preprocessing function that:

1. Reads the image from disk.
2. Decodes it as RGB.
3. Converts it to floats in `[0, 1]`.
4. Resizes to `IMG_SIZE × IMG_SIZE`.

The labels are kept as `int32` tensors of shape `(8, 8)` with values in `{0, 1, 2}`.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_image(path, label):
    # Read and decode image
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_image(img_bytes, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)  # scale to [0,1]
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    # Ensure static shape
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))

    # Cast labels to int32
    label = tf.cast(label, tf.int32)
    return img, label

def make_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)
    ds = ds.map(preprocess_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_paths, train_labels, shuffle=True)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)

## 7. Define the CNN model

The model:

- Takes an image of shape `(IMG_SIZE, IMG_SIZE, 3)`.
- Passes it through several convolution + pooling layers.
- Uses a dense layer to produce `8 × 8 × 3 = 192` logits.
- Reshapes the logits to `(8, 8, 3)`, giving a 3‑way classification for each LED.

We train with **sparse categorical cross‑entropy** over the 8×8 grid:

- Class `0` = off  
- Class `1` = red  
- Class `2` = blue

In [ ]:
def build_model(img_size=IMG_SIZE):
    inputs = keras.Input(shape=(img_size, img_size, 3))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    # 8x8x3 logits (3 classes per cell)
    x = layers.Dense(8 * 8 * 3)(x)
    outputs = layers.Reshape((8, 8, 3))(x)

    model = keras.Model(inputs, outputs, name="led_matrix_cnn")
    return model

model = build_model()
model.summary()

## 8. Compile and train

We use the Adam optimizer and sparse categorical cross‑entropy loss, treating the 8×8 grid as a set of 64 independent 3‑class classification problems.

In [ ]:
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss_fn,
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="cell_acc")]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

## 9. Visualize training and validate on a few samples

We can: 
- Plot training / validation loss curves.  
- Look at a few predictions and compare to the ground‑truth 64‑element arrays.

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history):
    hist = history.history
    epochs_range = range(1, len(hist["loss"]) + 1)

    plt.figure()
    plt.plot(epochs_range, hist["loss"], label="Train loss")
    plt.plot(epochs_range, hist["val_loss"], label="Val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.title("Training vs Validation Loss")
    plt.show()

plot_history(history)

In [ ]:
# Inspect predictions on a few validation samples
val_batch_images, val_batch_labels = next(iter(val_ds))
logits = model.predict(val_batch_images)
pred_classes = tf.argmax(logits, axis=-1).numpy()  # shape: (batch, 8, 8)

num_to_show = min(3, pred_classes.shape[0])
for i in range(num_to_show):
    true_board = val_batch_labels[i].numpy().astype(int).reshape(-1)
    pred_board = pred_classes[i].reshape(-1)
    print(f"Sample {i}:")
    print("  True board (64 elements):\n", true_board)
    print("  Pred board (64 elements):\n", pred_board)
    print()


## 10. Save the trained model

You can save the Keras model (`.keras` or `.h5`) and optionally convert it to TensorFlow Lite if needed.

In [ ]:
MODEL_PATH = "/content/led_matrix_cnn.keras"  # or .h5
model.save(MODEL_PATH)
print("Saved model to:", MODEL_PATH)

# OPTIONAL: convert to TFLite (uncomment if desired)
# converter = tf.lite.TFLiteConverter.from_keras_model(model)
# tflite_model = converter.convert()
# with open("/content/led_matrix_cnn.tflite", "wb") as f:
#     f.write(tflite_model)
# print("Saved TFLite model.")

## 11. Inference on a single new image

Given a path to a new LED matrix image, this helper function returns the predicted 64‑element array with values in `{0, 1, 2}`.

In [ ]:
def predict_board_from_image(model, img_path):
    img_bytes = tf.io.read_file(img_path)
    img = tf.image.decode_image(img_bytes, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))

    img_batch = tf.expand_dims(img, axis=0)  # shape (1, H, W, 3)
    logits = model(img_batch)
    pred = tf.argmax(logits, axis=-1).numpy()[0]  # shape (8, 8)
    return pred.reshape(-1)  # 64‑element array

# Example usage (update the path):
# example_image = image_paths[0]
# predicted_board = predict_board_from_image(model, example_image)
# print("Predicted board (64 elements):", predicted_board)